In [ ]:
# ── IMPORTS ──────────────────────────────────────────
import shap
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

print('✅ Imports OK !')

In [ ]:
# ── CHARGER ET PRÉPARER LA DATA ───────────────────────
dataset = fetch_ucirepo(id=544)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# Encodage
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

le = LabelEncoder()
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    df[col] = le.fit_transform(df[col])

le_target = LabelEncoder()
df['NObeyesdad'] = le_target.fit_transform(df['NObeyesdad'])

# Split
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('✅ Data prête !')

In [ ]:
# ── CHARGER LE MODÈLE LIGHTGBM ───────────────────────
with open('outputs/lightgbm.pkl', 'rb') as f:
    best_model = pickle.load(f)

print('✅ Modèle LightGBM chargé !')

In [ ]:
# ── CRÉER L'EXPLAINER SHAP ───────────────────────────
print('Calcul des SHAP values...')
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print('✅ SHAP values calculées !')

In [ ]:
# ── GRAPHIQUE 1 : Features les plus importantes ───────
print('📊 Graphique 1 — Feature Importance Global')
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    plot_type='bar',
    class_names=le_target.classes_,
    show=True
)

In [ ]:
# ── GRAPHIQUE 2 : Impact détaillé ────────────────────
print('📊 Graphique 2 — Impact détaillé par feature')
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    class_names=le_target.classes_,
    show=True
)

In [ ]:
# ── GRAPHIQUE 3 : Explication d'un seul patient ───────
print('📊 Graphique 3 — Explication patient #0')
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0][0],
        base_values=explainer.expected_value[0],
        data=X_test.iloc[0],
        feature_names=X_test.columns.tolist()
    )
)

In [ ]:
# ── RÉSUMÉ : Top 5 features ──────────────────────────
print('=' * 50)
print('RÉSUMÉ SHAP — Top 5 Features les plus importantes')
print('=' * 50)

mean_shap = np.abs(np.array(shap_values)).mean(axis=(0, 1))
feature_importance = dict(zip(X_test.columns, mean_shap))
sorted_features = sorted(feature_importance.items(),
                          key=lambda x: x[1], reverse=True)

for i, (feat, val) in enumerate(sorted_features[:5], 1):
    print(f'  {i}. {feat} : {val:.4f}')

print('=' * 50)
print('✅ Analyse SHAP terminée !')